In [1]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge

print('Training Solar Forecasting Model...')
df_solar = pd.read_csv('merged_nasa_pvgis_hourly.csv')
df_solar['timestamp'] = pd.to_datetime(df_solar['timestamp'])
df_solar['hour'] = df_solar['timestamp'].dt.hour
df_solar['month'] = df_solar['timestamp'].dt.month
df_solar['dayofyear'] = df_solar['timestamp'].dt.dayofyear

solar_features = [
    'NASA_temperature_C',
    'NASA_relative_humidity_pct',
    'NASA_wind_speed_m_s',
    'NASA_precipitation_mm_per_hour',
    'PVGIS_direct_irradiance_W_m2',
    'PVGIS_diffuse_irradiance_W_m2',
    'PVGIS_sun_height_deg',
    'hour',
    'month',
    'dayofyear',
]
X_solar = df_solar[solar_features]
y_solar = df_solar['PVGIS_PV_power_W_for_1kWp_system']

solar_model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
solar_model.fit(X_solar, y_solar)
joblib.dump(solar_model, 'solar_model.pkl')
print('Saved solar_model.pkl successfully!')

print('Training Building Load Forecasting Model...')
df_load = pd.read_csv('household_data_60min_singleindex.csv')
cols = [
    'utc_timestamp',
    'DE_KN_residential3_grid_import',
    'DE_KN_residential3_grid_export',
    'DE_KN_residential3_pv',
]
df_load_sub = df_load[cols].copy()
df_load_sub['utc_timestamp'] = pd.to_datetime(df_load_sub['utc_timestamp'])
df_load_sub = df_load_sub.sort_values('utc_timestamp').reset_index(drop=True)
df_load_sub[
    [
        'DE_KN_residential3_grid_import',
        'DE_KN_residential3_grid_export',
        'DE_KN_residential3_pv',
    ]
] = df_load_sub[
    [
        'DE_KN_residential3_grid_import',
        'DE_KN_residential3_grid_export',
        'DE_KN_residential3_pv',
    ]
].fillna(
    0
)

df_load_sub['total_load'] = (
    df_load_sub['DE_KN_residential3_grid_import']
    + df_load_sub['DE_KN_residential3_pv']
    - df_load_sub['DE_KN_residential3_grid_export']
)
df_load_sub['hour'] = df_load_sub['utc_timestamp'].dt.hour
df_load_sub['dayofweek'] = df_load_sub['utc_timestamp'].dt.dayofweek
df_load_sub['month'] = df_load_sub['utc_timestamp'].dt.month

df_load_sub['load_lag_24'] = df_load_sub['total_load'].shift(24)
df_load_sub['load_lag_48'] = df_load_sub['total_load'].shift(48)
df_load_sub['load_lag_168'] = df_load_sub['total_load'].shift(168)
df_load_clean = df_load_sub.dropna().reset_index(drop=True)

load_features = [
    'hour',
    'dayofweek',
    'month',
    'load_lag_24',
    'load_lag_48',
    'load_lag_168',
]
X_load = df_load_clean[load_features]
y_load = df_load_clean['total_load']

load_model = Ridge(alpha=1.0)
load_model.fit(X_load, y_load)
joblib.dump(load_model, 'load_model.pkl')
print('Saved load_model.pkl successfully!')

Training Solar Forecasting Model...
Saved solar_model.pkl successfully!
Training Building Load Forecasting Model...
Saved load_model.pkl successfully!
